In [4]:
import h5py
import glob
import numpy as np

# Find all H5 files
h5_files = glob.glob(r'MillionSongSubset/*/*/*/*.h5')
print(f"Total files: {len(h5_files)}")

# Explore one file's structure
with h5py.File(h5_files[0], 'r') as f:
    def print_structure(name, obj):
        print(name)
    f.visititems(print_structure)

Total files: 10000
analysis
analysis/bars_confidence
analysis/bars_start
analysis/beats_confidence
analysis/beats_start
analysis/sections_confidence
analysis/sections_start
analysis/segments_confidence
analysis/segments_loudness_max
analysis/segments_loudness_max_time
analysis/segments_loudness_start
analysis/segments_pitches
analysis/segments_start
analysis/segments_timbre
analysis/songs
analysis/tatums_confidence
analysis/tatums_start
metadata
metadata/artist_terms
metadata/artist_terms_freq
metadata/artist_terms_weight
metadata/similar_artists
metadata/songs
musicbrainz
musicbrainz/artist_mbtags
musicbrainz/artist_mbtags_count
musicbrainz/songs


In [7]:
# Enhanced feature extraction from Million Song Dataset
def load_song_data(h5_file):
    with h5py.File(h5_file, 'r') as f:
        # ===== METADATA FIELDS =====
        title = f['metadata/songs']['title'][0].decode('utf-8')
        artist = f['metadata/songs']['artist_name'][0].decode('utf-8')
        artist_id = f['metadata/songs']['artist_id'][0].decode('utf-8')
        song_id = f['metadata/songs']['song_id'][0].decode('utf-8')
        release = f['metadata/songs']['release'][0].decode('utf-8')
        artist_location = f['metadata/songs']['artist_location'][0].decode('utf-8')
    
        
        # ===== POPULARITY/HOTNESS FEATURES =====
        artist_familiarity = f['metadata/songs']['artist_familiarity'][0]
        artist_hotttnesss = f['metadata/songs']['artist_hotttnesss'][0]
        song_hotttnesss = f['metadata/songs']['song_hotttnesss'][0]
        
        # ===== AUDIO ANALYSIS FEATURES =====
        # Segments: timbre (12-dim) and pitches (12-dim)
        timbre = f['analysis/segments_timbre'][()]
        pitches = f['analysis/segments_pitches'][()]
        segments_confidence = f['analysis/segments_confidence'][()] if 'analysis/segments_confidence' in f else np.array([])
        
        # Loudness features
        loudness_max = f['analysis/segments_loudness_max'][()] if 'analysis/segments_loudness_max' in f else np.array([])
        loudness_max_time = f['analysis/segments_loudness_max_time'][()] if 'analysis/segments_loudness_max_time' in f else np.array([])
        loudness_start = f['analysis/segments_loudness_start'][()] if 'analysis/segments_loudness_start' in f else np.array([])
        
        # Time-based features: bars, beats, sections, tatums
        bars_confidence = f['analysis/bars_confidence'][()] if 'analysis/bars_confidence' in f else np.array([])
        bars_start = f['analysis/bars_start'][()] if 'analysis/bars_start' in f else np.array([])
        
        beats_confidence = f['analysis/beats_confidence'][()] if 'analysis/beats_confidence' in f else np.array([])
        beats_start = f['analysis/beats_start'][()] if 'analysis/beats_start' in f else np.array([])
        
        sections_confidence = f['analysis/sections_confidence'][()] if 'analysis/sections_confidence' in f else np.array([])
        sections_start = f['analysis/sections_start'][()] if 'analysis/sections_start' in f else np.array([])
        
        tatums_confidence = f['analysis/tatums_confidence'][()] if 'analysis/tatums_confidence' in f else np.array([])
        tatums_start = f['analysis/tatums_start'][()] if 'analysis/tatums_start' in f else np.array([])
        
        # ===== ARTIST TERMS (Tags/Genres) =====
        artist_terms_data = f['metadata/artist_terms'][()] if 'metadata/artist_terms' in f else np.array([])
        artist_terms = [term.decode('utf-8') if isinstance(term, bytes) else term 
                       for term in artist_terms_data]
        
        artist_terms_freq = f['metadata/artist_terms_freq'][()] if 'metadata/artist_terms_freq' in f else np.array([])
        artist_terms_weight = f['metadata/artist_terms_weight'][()] if 'metadata/artist_terms_weight' in f else np.array([])
        
        # ===== SIMILAR ARTISTS =====
        similar_artists_data = f['metadata/similar_artists'][()]
        similar_artists = [aid.decode('utf-8') if isinstance(aid, bytes) else aid 
                        for aid in similar_artists_data]
        
        # ===== MUSICBRAINZ TAGS =====
        mbtags = f['musicbrainz/artist_mbtags'][()] if 'musicbrainz/artist_mbtags' in f else np.array([])
        artist_mbtags = [tag.decode('utf-8') if isinstance(tag, bytes) else tag 
                        for tag in mbtags]
        
        mbtags_count = f['musicbrainz/artist_mbtags_count'][()] if 'musicbrainz/artist_mbtags_count' in f else np.array([])
        
        return {
            # Basic metadata
            'song_id': song_id,
            'title': title,
            'artist': artist,
            'artist_id': artist_id,
            'release': release,
            'artist_location': artist_location,
            
            # Popularity metrics
            'artist_familiarity': artist_familiarity,
            'artist_hotttnesss': artist_hotttnesss,
            'song_hotttnesss': song_hotttnesss,
            
            # Audio features: timbre and pitches
            'timbre': timbre,
            'pitches': pitches,
            'segments_confidence': segments_confidence,
            
            # Loudness features
            'loudness_max': loudness_max,
            'loudness_max_time': loudness_max_time,
            'loudness_start': loudness_start,
            
            # Temporal structure
            'bars_confidence': bars_confidence,
            'bars_start': bars_start,
            'beats_confidence': beats_confidence,
            'beats_start': beats_start,
            'sections_confidence': sections_confidence,
            'sections_start': sections_start,
            'tatums_confidence': tatums_confidence,
            'tatums_start': tatums_start,
            
            # Artist terms and tags
            'artist_terms': artist_terms,
            'artist_terms_freq': artist_terms_freq,
            'artist_terms_weight': artist_terms_weight,
            'artist_mbtags': artist_mbtags,
            'artist_mbtags_count': mbtags_count,
            
            # Similar artists
            'similar_artists': similar_artists,
        }


In [10]:
# Test on single file - display enhanced features
song = load_song_data(h5_files[0])

print(f"  Song: {song['title']}")
print(f"  Artist: {song['artist']}")
print(f"  Release: {song['release']}")
print(f"  Location: {song['artist_location']}")
print(f"  Song Hotness: {song['song_hotttnesss']:.3f}")
print(f"  Artist Familiarity: {song['artist_familiarity']:.3f}")
print(f"  Artist Hotness: {song['artist_hotttnesss']:.3f}")
print(f"  Timbre shape: {song['timbre'].shape}")
print(f"  Pitches shape: {song['pitches'].shape}")
print(f"  Segments confidence: {len(song['segments_confidence'])} segments")

print(f"  Loudness max segments: {len(song['loudness_max'])}")
print(f"  Loudness start segments: {len(song['loudness_start'])}")

print(f"  Bars: {len(song['bars_start'])} bars")
print(f"  Beats: {len(song['beats_start'])} beats")
print(f"  Sections: {len(song['sections_start'])} sections")
print(f"  Tatums: {len(song['tatums_start'])} tatums")

print(f"  Artist terms: {len(song['artist_terms'])} tags - {', '.join(song['artist_terms'][:5])}")
print(f"  MusicBrainz tags: {len(song['artist_mbtags'])} tags - {', '.join(song['artist_mbtags'][:3])}")
print(f"  Similar artists: {len(song['similar_artists'])} artists")

  Song: I Didn't Mean To
  Artist: Casual
  Release: Fear Itself
  Location: California - LA
  Song Hotness: 0.602
  Artist Familiarity: 0.582
  Artist Hotness: 0.402
  Timbre shape: (971, 12)
  Pitches shape: (971, 12)
  Segments confidence: 971 segments
  Loudness max segments: 971
  Loudness start segments: 971
  Bars: 83 bars
  Beats: 344 beats
  Sections: 10 sections
  Tatums: 688 tatums
  Artist terms: 37 tags - hip hop, underground rap, g funk, alternative rap, gothic rock
  MusicBrainz tags: 0 tags - 
  Similar artists: 100 artists


In [11]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load data from first N files
N = 10000
songs_data = []

def aggregate_features(arr, prefix=''):
    """Aggregate array features into scalar statistics"""
    if len(arr) == 0:
        return {}
    
    stats = {}
    if prefix:
        stats[f'{prefix}_mean'] = arr.mean()
        stats[f'{prefix}_std'] = arr.std()
        stats[f'{prefix}_max'] = arr.max()
        stats[f'{prefix}_min'] = arr.min()
    else:
        stats['mean'] = arr.mean()
        stats['std'] = arr.std()
        stats['max'] = arr.max()
        stats['min'] = arr.min()
    
    return stats

for h5_file in tqdm(h5_files[:N], desc="Loading songs"):
    try:
        song = load_song_data(h5_file)
        
        # Base song info
        row = {
            'song_id': song['song_id'],
            'title': song['title'],
            'artist': song['artist'],
            'artist_id': song['artist_id'],
            'release': song['release'],
            'artist_location': song['artist_location'],
            'artist_familiarity': song['artist_familiarity'],
            'artist_hotttnesss': song['artist_hotttnesss'],
            'song_hotttnesss': song['song_hotttnesss'],
        }
        
        # Audio features: timbre and pitches (mean across segments)
        row['timbre_mean'] = song['timbre'].mean(axis=0)
        row['timbre_std'] = song['timbre'].std(axis=0)
        row['pitches_mean'] = song['pitches'].mean(axis=0)
        row['pitches_std'] = song['pitches'].std(axis=0)
        
        # Aggregate segment-level features
        if len(song['segments_confidence']) > 0:
            row['segments_confidence_mean'] = song['segments_confidence'].mean()
            row['segments_confidence_std'] = song['segments_confidence'].std()
        
        # Loudness features
        if len(song['loudness_max']) > 0:
            row['loudness_max_mean'] = song['loudness_max'].mean()
            row['loudness_max_std'] = song['loudness_max'].std()
            row['loudness_max_max'] = song['loudness_max'].max()
            row['loudness_max_min'] = song['loudness_max'].min()
            row['loudness_start_mean'] = song['loudness_start'].mean()
            row['loudness_start_std'] = song['loudness_start'].std()
        
        # Temporal features: bars, beats, sections, tatums (count + confidence stats)
        row['num_bars'] = len(song['bars_start'])
        row['num_beats'] = len(song['beats_start'])
        row['num_sections'] = len(song['sections_start'])
        row['num_tatums'] = len(song['tatums_start'])
        
        if len(song['bars_confidence']) > 0:
            row['bars_confidence_mean'] = song['bars_confidence'].mean()
        if len(song['beats_confidence']) > 0:
            row['beats_confidence_mean'] = song['beats_confidence'].mean()
        if len(song['sections_confidence']) > 0:
            row['sections_confidence_mean'] = song['sections_confidence'].mean()
        if len(song['tatums_confidence']) > 0:
            row['tatums_confidence_mean'] = song['tatums_confidence'].mean()
        
        # Tempo derived from beats (beats per second)
        if len(song['beats_start']) > 1:
            beat_intervals = np.diff(song['beats_start'])
            row['tempo_estimate'] = 1.0 / beat_intervals.mean() if beat_intervals.mean() > 0 else 0
        else:
            row['tempo_estimate'] = 0
        
        # Artist terms (as string, top 3)
        top_terms = sorted(
            zip(song['artist_terms'], song['artist_terms_weight']),
            key=lambda x: x[1],
            reverse=True
        )[:3]
        row['top_artist_terms'] = ', '.join([t[0] for t in top_terms])
        
        # MusicBrainz tags
        row['num_mbtags'] = len(song['artist_mbtags'])
        if len(song['artist_mbtags']) > 0:
            top_mbtags = song['artist_mbtags'][:3]
            row['top_mbtags'] = ', '.join(top_mbtags)
        
        # Similar artists count
        row['num_similar_artists'] = len(song['similar_artists'])
        
        songs_data.append(row)
    except Exception as e:
        print(f"Error loading {h5_file}: {e}")

df = pd.DataFrame(songs_data)
print(f"Dataset shape: {df.shape}")
print(f"\nDataset columns: {list(df.columns)}")
df.head()


Loading songs: 100%|██████████| 10000/10000 [10:12<00:00, 16.32it/s]


Dataset shape: (10000, 34)

Dataset columns: ['song_id', 'title', 'artist', 'artist_id', 'release', 'artist_location', 'artist_familiarity', 'artist_hotttnesss', 'song_hotttnesss', 'timbre_mean', 'timbre_std', 'pitches_mean', 'pitches_std', 'segments_confidence_mean', 'segments_confidence_std', 'loudness_max_mean', 'loudness_max_std', 'loudness_max_max', 'loudness_max_min', 'loudness_start_mean', 'loudness_start_std', 'num_bars', 'num_beats', 'num_sections', 'num_tatums', 'bars_confidence_mean', 'beats_confidence_mean', 'sections_confidence_mean', 'tatums_confidence_mean', 'tempo_estimate', 'top_artist_terms', 'num_mbtags', 'num_similar_artists', 'top_mbtags']


,song_id,title,artist,artist_id,release,artist_location,artist_familiarity,artist_hotttnesss,song_hotttnesss,timbre_mean,...,num_tatums,bars_confidence_mean,beats_confidence_mean,sections_confidence_mean,tatums_confidence_mean,tempo_estimate,top_artist_terms,num_mbtags,num_similar_artists,top_mbtags
0,SOMZWCG12A8C13C480,I Didn't Mean To,Casual,ARD7TVE1187B99BFB1,Fear Itself,California - LA,0.581794,0.401998,0.602120,"[41.51277754891864, 15.662936148300714, -5.789...",...,688,0.172096,0.611462,0.477100,0.455733,1.573880,"hip hop, underground rap, g funk",0,100,NaN
1,SOCIWDW12A8C13D406,Soul Deep,The Box Tops,ARMJAGH1187FB546F3,Dimensions,"Memphis, TN",0.630630,0.417500,NaN,"[43.07103636363632, -4.035390909090912, 23.572...",...,591,0.122562,0.730807,0.491667,0.616008,2.025150,"blue-eyed soul, pop rock, blues-rock",1,100,classic pop and rock
2,SOXVLOJ12AB0189215,Amor De Cabaret,Sonora Santanera,ARKRRTF1187B9984DA,Las Numero 1 De La Sonora Santanera,,0.487357,0.343428,NaN,"[45.130814946619225, -76.8233256227758, 50.787...",...,582,0.430550,0.430550,0.451875,0.332560,1.678399,"salsa, cumbia, tejano",0,100,NaN
3,SONHOTT12A8C13493C,Something Girls,Adam Ant,AR7G5I41187FB4CE6C,Friend Or Foe,"London, England",0.630382,0.454231,NaN,"[45.800255785627286, 41.148986601705204, 57.59...",...,924,0.118609,0.621877,0.325818,0.260192,1.988176,"pop rock, new wave, dance rock",3,100,"uk, british, english"
4,SOFSOCN12A8C143F5D,Face the Ashes,Gob,ARXR32B1187FB57099,Muertos Vivos,,0.651046,0.401724,0.604501,"[50.25155423476959, 27.845583952451705, 47.091...",...,887,0.127936,0.435171,0.613333,0.257529,2.166444,"pop punk, ska punk, breakcore",0,100,NaN


In [12]:
# Save dataset to CSV and pickle formats
import os

output_dir = 'processed_data'
os.makedirs(output_dir, exist_ok=True)

# Prepare data for CSV (convert arrays to strings for CSV compatibility)
df_csv = df.copy()

# Convert numpy arrays to strings or lists
for col in df_csv.columns:
    if df_csv[col].dtype == object:
        # Check if contains numpy arrays
        try:
            if isinstance(df_csv[col].iloc[0], np.ndarray):
                # Convert arrays to semicolon-separated strings
                df_csv[col] = df_csv[col].apply(lambda x: ';'.join(map(str, x.flatten())) if isinstance(x, np.ndarray) else str(x))
        except:
            pass

# Save as CSV
csv_path = os.path.join(output_dir, f'music_dataset_{N}_songs.csv')
df_csv.to_csv(csv_path, index=False)
print(f"✅ CSV saved: {csv_path}")

# Save as Pickle (preserves numpy arrays)
pkl_path = os.path.join(output_dir, f'music_dataset_{N}_songs.pkl')
df.to_pickle(pkl_path)
print(f"✅ Pickle saved: {pkl_path}")

# Create metadata file
metadata = {
    'total_songs': len(df),
    'features': list(df.columns),
    'feature_count': len(df.columns),
    'date_created': pd.Timestamp.now(),
    'feature_categories': {
        'metadata': ['song_id', 'title', 'artist', 'artist_id', 'release', 'artist_location', 'year', 'duration'],
        'popularity': ['artist_familiarity', 'artist_hotttnesss', 'song_hotttnesss'],
        'audio_features': [col for col in df.columns if any(x in col for x in ['timbre', 'pitches', 'loudness', 'segment'])],
        'temporal_structure': [col for col in df.columns if any(x in col for x in ['bars', 'beats', 'sections', 'tatums', 'tempo'])],
        'artist_info': [col for col in df.columns if any(x in col for x in ['artist_term', 'mbtag', 'similar'])]
    }
}

import json
metadata_path = os.path.join(output_dir, 'dataset_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump({k: v if k != 'date_created' else str(v) for k, v in metadata.items()}, f, indent=2)
print(f"✅ Metadata saved: {metadata_path}")

print(f"\n📂 All files saved in '{output_dir}/' directory")


✅ CSV saved: processed_data\music_dataset_10000_songs.csv
✅ Pickle saved: processed_data\music_dataset_10000_songs.pkl
✅ Metadata saved: processed_data\dataset_metadata.json

📂 All files saved in 'processed_data/' directory
